from re import compile
utok = np.array([b'\n !"%&\'()*+,-./0123456789:<=>@[]^_abcdefghijklmnopqrstuvwxyz{|}~']).view('B')
utok.tofile(f'py.{len(utok)}tok')
tok = np.full(2<<32-1,255,'B')
tok[tok] = range(len(utok))
tok[ord('\t')] = tok[ord(' ')]
tok[ord('A'):ord('Z')+1] = tok[ord('a'):ord('z')+1]
sub = {compile(r'\s*(#.*?)?\n'):',', compile('\n{2,}'):'\n', compile(r'\s+,'):',', '    ':' '}
a = open('py.txt', errors='ignore')
o = open(f'py.{len(utok)}', 'wb')
while _ := a.read(1<<24):
    for i,j in sub.items():
        _ = i.sub(j,_) if hasattr(i,'sub') else _.replace(i,j)
    _ = tok[np.array([_]).view('I')]
    _[_!=255].tofile(o)

In [ ]:
import numpy as np
from heapq import heapify, heappushpop, heappop
from json import dump
def gram(a, n):
    a = np.sort(np.lib.stride_tricks.sliding_window_view(a,n).view(f'S{n}')[:,0])
    i, = (a[:-1]!=a[1:]).nonzero()
    c = np.empty(len(i)+1, 'i')
    c[0] = i[0]+1
    c[-1] = len(a)-1-i[-1]
    np.subtract(i[1:], i[:-1], c[1:-1], casting='unsafe')
    I = np.empty(len(i)+1, f'S{n}')
    np.take(a, i, out=I[:-1])
    I[-1] = a[i[-1]+1]
    return I, c
C = {i.item():c.item()for i,c in zip(*gram(np.memmap('../../py.txt',mode='r'),16))}
H = [(c,-len(i),i)for i,c in C.items()]
heapify(H)
h = None
while len(C) > 65536-64:
    c,l,i = heappushpop(H,h) if h else heappop(H)
    h = None
    if c == C[i]:
        del C[i]
        if l < -2:
            i = i[:-1]
            c += C.get(i,0)
            C[i] = c
            h = c,l+1,i
dump({c.decode():i for c,i in C.items()}, open('../../py65536py16.json','w'), separators=',:')

In [ ]:
while len(C) > 256-64:
    c,l,i = heappushpop(H,h) if h else heappop(H)
    h = None
    if c == C[i]:
        del C[i]
        if l < -2:
            i = i[:-1]
            c += C.get(i,0)
            C[i] = c
            h = c,l+1,i
dump({c.decode():i for c,i in C.items()}, open('../../py256py16.json','w'), separators=',:')